1-ая задача замёрджена в 02_geography.ipynb (дописал 7-ой шаг)

2-я задача

In [1]:
import folium
import geopandas as gpd
import pandas as pd

def add_candidates_layer(geojson_path, layer_name="Кандидаты"):
    gdf = gpd.read_file(geojson_path)
    required_fields = {
        'address': 'Адрес не указан',
        'price_rub_per_month': None,
        'area_m2': None,
        'expected_coverage': None
    }
    for field, default in required_fields.items():
        if field not in gdf.columns:
            gdf[field] = default

    group = folium.FeatureGroup(name=layer_name)

    for _, row in gdf.iterrows():
        popup_html = f"""
        <div style="font-family: sans-serif; font-size: 12px; line-height: 1.4;">
            <b>🏢 {row['address']}</b><br>
            <table style="margin-top: 5px;">
                <tr><td><b>Цена аренды:</b></td><td>{_fmt(row['price_rub_per_month'])}</td></tr>
                <tr><td><b>Площадь:</b></td><td>{_fmt(row['area_m2'], suffix=' м²')}</td></tr>
                <tr><td><b>Охват населения:</b></td><td>{_fmt(row['expected_coverage'], suffix=' чел.')}</td></tr>
            </table>
        </div>
        """

        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=folium.Popup(popup_html, max_width=250),
            icon=folium.Icon(color='red', icon='star')
        ).add_to(group)

    return group


def _fmt(value, suffix=''):
    if pd.isna(value) or value is None:
        return '—'
    if isinstance(value, (int, float)):
        return f"{value:,.0f}{suffix}".replace(",", " ")
    return f"{value}{suffix}"


3-я задача

In [2]:
import networkx as nx
from scipy.spatial import cKDTree
import numpy as np
from pyvis.network import Network
from pathlib import Path

def build_accessibility_graph(
    pvz_gdf,
    candidates_gdf,
    output_path="reports/accessibility_graph.html",
    nearest_k=3,
    use_projected_crs=True
):

    if use_projected_crs:
        target_crs = "EPSG:32646"
        pvz_geom = pvz_gdf.to_crs(target_crs).geometry
        cand_geom = candidates_gdf.to_crs(target_crs).geometry
    else:
        pvz_geom = pvz_gdf.geometry
        cand_geom = candidates_gdf.geometry

    pvz_coords = np.array([[pt.x, pt.y] for pt in pvz_geom])
    cand_coords = np.array([[pt.x, pt.y] for pt in cand_geom])

    tree = cKDTree(pvz_coords)
    distances, indices = tree.query(cand_coords, k=nearest_k)

    G = nx.Graph()

    for i, (_, row) in enumerate(pvz_gdf.iterrows()):
        G.add_node(
            f"pvz_{i}",
            label=row.get("address", f"PVZ {i}"),
            node_type="pvz",
            title=f"Текущий ПВЗ<br>{row.get('address', '')}"
        )

    for j, (_, row) in enumerate(candidates_gdf.iterrows()):
        G.add_node(
            f"cand_{j}",
            label=row.get("address", f"Candidate {j}"),
            node_type="candidate",
            title=(
                f"<b>Кандидат</b><br>"
                f"Адрес: {row.get('address', '—')}<br>"
                f"Цена: {row.get('price_rub_per_month', '—')} ₽/мес<br>"
                f"Площадь: {row.get('area_m2', '—')} м²"
            )
        )

    for j in range(len(candidates_gdf)):
        for k_idx in range(nearest_k):
            pvz_i = indices[j, k_idx]
            dist = distances[j, k_idx]
            G.add_edge(
                f"cand_{j}", f"pvz_{pvz_i}",
                weight=round(dist, 1),
                title=f"Расстояние: {dist:.1f} м"
            )

    net = Network(height="700px", width="100%", bgcolor="#ffffff", font_color="black")
    net.from_nx(G)

    for node in net.nodes:
        node_id = node["id"]
        nx_node = G.nodes[node_id]
        node_type = nx_node.get("node_type", "pvz")

        if node_type == "pvz":
            node["color"] = "#aaaaaa"
            node["size"] = 8
        else:
            node["color"] = "#e63946"
            node["size"] = 20

        node["label"] = nx_node.get("label", node_id)
        node["title"] = nx_node.get("title", "")

    net.set_edge_smooth("dynamic")
    for edge in net.edges:
        edge["color"] = {"color": "#6666ff", "opacity": 0.7}
        edge["width"] = 1.5

    net.set_options("""
    var options = {
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -3000,
          "centralGravity": 0.3,
          "springLength": 150,
          "springConstant": 0.04,
          "damping": 0.09
        }
      }
    }
    """)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    net.write_html(str(output_path), open_browser=False, notebook=False)
    print(f"Граф сохранён в {output_path}")
    return net

In [3]:
from shapely.geometry import Point

BASE_DIR = Path.cwd().parent
processed_dir = BASE_DIR / "data" / "processed"

pvz_df = pd.read_csv(processed_dir / "pvz_krasnoyarsk.csv")
pvz_gdf = gpd.GeoDataFrame(
    pvz_df,
    geometry=[Point(xy) for xy in zip(pvz_df.longitude, pvz_df.latitude)],
    crs="EPSG:4326"
)

candidates_gdf = gpd.read_file(processed_dir / "candidates_v2_smart.geojson")

print(f"Загружено ПВЗ: {len(pvz_gdf)}, кандидатов: {len(candidates_gdf)}")

Загружено ПВЗ: 420, кандидатов: 5


In [4]:
net = build_accessibility_graph(
    pvz_gdf,
    candidates_gdf,
    output_path=BASE_DIR / "reports" / "accessibility_graph.html",
    nearest_k=3
)

Граф сохранён в /Users/aivi/Documents/Programming/Git_repositories/Optimization-of-Ozon-pick-up-points-in-Krasnoyarsk/reports/accessibility_graph.html


4-я задача

In [5]:
def evaluate_candidate_distances(pvz_gdf, candidates_gdf, nearest_k=3):
    target_crs = "EPSG:32646"
    pvz_geom = pvz_gdf.to_crs(target_crs).geometry
    cand_geom = candidates_gdf.to_crs(target_crs).geometry

    pvz_coords = np.array([[pt.x, pt.y] for pt in pvz_geom])
    cand_coords = np.array([[pt.x, pt.y] for pt in cand_geom])

    tree = cKDTree(pvz_coords)
    distances, indices = tree.query(cand_coords, k=nearest_k)

    all_dists = distances.flatten()

    metrics = {
        "mean": np.mean(all_dists),
        "median": np.median(all_dists),
        "min": np.min(all_dists),
        "max": np.max(all_dists)
    }
    return metrics

metrics = evaluate_candidate_distances(pvz_gdf, candidates_gdf, nearest_k=3)

print("=== МЕТРИКИ БЛИЗОСТИ К СЕТИ ПВЗ ===")
print(f"Среднее расстояние до 3 ближайших ПВЗ: {metrics['mean']:.0f} м")
print(f"Медианное расстояние: {metrics['median']:.0f} м")
print(f"Минимальное расстояние: {metrics['min']:.0f} м")
print(f"Максимальное расстояние: {metrics['max']:.0f} м")

=== МЕТРИКИ БЛИЗОСТИ К СЕТИ ПВЗ ===
Среднее расстояние до 3 ближайших ПВЗ: 873 м
Медианное расстояние: 825 м
Минимальное расстояние: 454 м
Максимальное расстояние: 1615 м
